In [ ]:
import json
import os
try:
    from backend.utils import load_json
except ModuleNotFoundError:
    print("Updating working directory....")
    os.chdir("../")
    # print(os.getcwd())
    from backend.utils import load_json
    
from tqdm import tqdm 
from backend.utils import load_response_dict
from backend.dataset.utils import organize_dataset
from backend.nli_labels import ExNLILabel
import math

import matplotlib.pyplot as plt
import numpy as np

from backend.uncertainty.utils import remove_invalid, calculate_probs, find_relevant_logprobs, sort_correct_wrong
from backend.uncertainty.metrics import calculate_uncertainty
import itertools

Updating working directory....


In [ ]:
set_logprob_scope = "first_token"

In [ ]:
# load dataset
from pathlib import Path


dataset = load_json("data/contract-nli/test.json")
dataset = organize_dataset(dataset)

output_dir = Path("out/responses/logprobs")
model_dirs = [p for p in output_dir.iterdir() if p.is_dir()]
print(model_dirs)

def match_substr(substr, exclude=None):
    return [p for p in model_dirs if (substr in str(p) and (exclude is None or exclude not in str(p)))]

gemma4 = match_substr("gemma4")
qwen35 = match_substr("qwen3.5")
qwen36 = match_substr("qwen3.6", exclude="nothink")
ministral3 = match_substr("ministral-3")

by_family = [gemma4, qwen35, qwen36, ministral3]
print(by_family)

all_fams = gemma4 + qwen35 + qwen36 + ministral3

response_dirs = {}
for md in all_fams:
    model = md.name
    model_dir = (md) / "narendra_nli_nonbinary_concise"
    print(list(model_dir.iterdir()))
    # point3 = [p for p in model_dir.iterdir() if (p.is_dir() and "0.3" in str(p))][0]
    seed_dirs = [str(p) for p in model_dir.iterdir() if (p.is_dir() and "seed" in str(p))]
    response_dirs[model] = (seed_dirs, "classification")
# print(response_dirs)

[PosixPath('out/responses/logprobs/ministral-3_14b'), PosixPath('out/responses/logprobs/gemma4_e2b'), PosixPath('out/responses/logprobs/ministral-3_8b'), PosixPath('out/responses/logprobs/gemma4_26b'), PosixPath('out/responses/logprobs/ministral-3_3b'), PosixPath('out/responses/logprobs/qwen3.5_4b'), PosixPath('out/responses/logprobs/gemma4_e4b'), PosixPath('out/responses/logprobs/qwen3.6_35b'), PosixPath('out/responses/logprobs/qwen3.5_4b_think'), PosixPath('out/responses/logprobs/qwen3.5_27b'), PosixPath('out/responses/logprobs/qwen3.5_9b')]
[[PosixPath('out/responses/logprobs/gemma4_e2b'), PosixPath('out/responses/logprobs/gemma4_26b'), PosixPath('out/responses/logprobs/gemma4_e4b')], [PosixPath('out/responses/logprobs/qwen3.5_4b'), PosixPath('out/responses/logprobs/qwen3.5_4b_think'), PosixPath('out/responses/logprobs/qwen3.5_27b'), PosixPath('out/responses/logprobs/qwen3.5_9b')], [PosixPath('out/responses/logprobs/qwen3.6_35b')], [PosixPath('out/responses/logprobs/ministral-3_14b'

In [ ]:


    # load and process responses

model_responses = {}
by_seed = {}
for model, (response_dir, structure) in response_dirs.items():
    by_seed[model] = {}
    correct, wrong = {}, {}
    for fp in response_dir:
        try:
            responses = load_response_dict(fp)
        except FileNotFoundError:
            print("File not found")
            continue
        # add probabilities to responses
        # calculate_probs(responses)
        print(fp)
        responses = remove_invalid(responses)

        find_relevant_logprobs(responses, relevant_scope=set_logprob_scope, structure=structure)
        calculate_uncertainty(responses)

        # find correct/wrong
        c, w = sort_correct_wrong(dataset, responses)

        # complicated indexing to save on boilerplating 
        for big_dict, sub_dict in [(correct, c), (wrong, w)]:
            if len(big_dict) == 0:
                for k,v in sub_dict.items():
                    big_dict[k] = v
            else:
                for k,v in big_dict.items():
                    big_dict[k] = v + sub_dict[k]
        
        seed = int(fp.split("/")[-1][4:])
        by_seed[model][seed] = {
            "correct": c,
            "wrong": w
        }
    
    model_responses[model] = {
        "correct": correct,
        "wrong": wrong
    }

Loading responses:   0%|          | 0/123 [00:00<?, ?it/s]

Loading responses: 100%|██████████| 123/123 [00:01<00:00, 70.97it/s]


out/responses/logprobs/gemma4_e2b/narendra_nli_nonbinary_concise/seed0
446 nda-15
Removed 1 invalid answers!


Probabilities:  42%|████▏     | 52/123 [00:00<00:00, 513.39it/s]

[([113, 114], '"'), ([113], '"')]
[([111, 112], 'Mentioned'), ([111], 'Mention')]
[([110, 111, 112], 'NotMentioned'), ([110, 111], 'NotMention'), ([110], 'Not')]
[([109, 110, 111, 112], '"NotMentioned'), ([109, 110, 111], '"NotMention'), ([109, 110], '"Not'), ([109], '"')]
[([108, 109, 110, 111, 112], '":"NotMentioned'), ([108, 109, 110, 111], '":"NotMention'), ([108, 109, 110], '":"Not'), ([108, 109], '":"'), ([108], '":')]
MATCH
[([4, 5], 'Mentioned'), ([4], 'Mention')]
MATCH
[([129, 130], '"'), ([129], '"')]
[([127, 128], 'ailment'), ([127], 'ail')]
[([126, 127, 128], 'Entailment'), ([126, 127], 'Entail'), ([126], 'Ent')]
[([125, 126, 127, 128], '"Entailment'), ([125, 126, 127], '"Entail'), ([125, 126], '"Ent'), ([125], '"')]
[([124, 125, 126, 127, 128], '":"Entailment'), ([124, 125, 126, 127], '":"Entail'), ([124, 125, 126], '":"Ent'), ([124, 125], '":"'), ([124], '":')]
MATCH
[([4, 5], 'ailment'), ([4], 'ail')]
MATCH
[([127, 128], 'ailment'), ([127], 'ail')]
[([126, 127, 128], 'En

...

Loading responses: 100%|██████████| 123/123 [00:02<00:00, 56.85it/s]


out/responses/logprobs/gemma4_26b/narendra_nli_nonbinary_concise/seed0
18 nda-7
543 nda-16
296 nda-7
1 nda-11
Removed 4 invalid answers!


Probabilities:   0%|          | 0/123 [00:00<?, ?it/s]


[([325, 326], 'ailment'), ([325], 'ail')]
[([324, 325, 326], 'Entailment'), ([324, 325], 'Entail'), ([324], 'Ent')]
[([320, 321], 'ailment'), ([320], 'ail')]
[([319, 320, 321], 'Entailment'), ([319, 320], 'Entail'), ([319], 'Ent')]
[([313, 314], ''), ([313], '')]
[([300, 301], ''), ([300], '')]
[([275, 276], ''), ([275], '')]
[([254, 255], ''), ([254], '')]
[([241, 242], ''), ([241], '')]
[([222, 223], ''), ([222], '')]
[([207, 208], ''), ([207], '')]
[([184, 185], ''), ([184], '')]
[([161, 162], ''), ([161], '')]
[([137, 138], ':"'), ([137], ':')]
[([129, 130], ''), ([129], '')]
[([106, 107], '":"'), ([106], '":')]
[([94, 95], ''), ([94], '')]
[([72, 73], ''), ([72], '')]
[([51, 52], ''), ([51], '')]
[([34, 35], ':"'), ([34], ':')]
[([28, 29], ''), ([28], '')]
Doc and hypo: 552 nda-10
'NoneType' object is not subscriptable


TypeError: 

In [ ]:
by_seed["gemma4_e2b"][0]["correct"]["NotMentioned"][0]["uncertainty"]

{'entropy': np.float64(0.7322386304826939),
 'average_logprob': np.float64(0.5579672302029999),
 'perplexity': np.float64(1.7471174006868007),
 'maximum_logprob': np.float64(21.30946159362793),
 'response_improbability': np.float64(1.0)}

In [ ]:
   
# pick model
sorted_model_responses = {}
for k,v in model_responses.items():
    sorted_model_responses[k] = {}

    model_dict = model_responses[k]

    # sort correct/wrong properly, by class
    correct = {}
    wrong = {}
    for cw_dict, cw_str in [[correct, "correct"], [wrong, "wrong"]]:
        ld = model_dict[cw_str]
        for label in ld.keys():
            if label not in cw_dict:
                cw_dict[label] = []
            for el in ld[label]:
                cw_dict[label] += [el["uncertainty"]["perplexity"]]

        sorted_model_responses[k][cw_str] = cw_dict
    
# # plot graph
# ax, fig = boxplot_analysis(correct, wrong, metric="Perplexity", title=f"Perplexity for GPT-OSS 20B", save_file=None)
# fig.show()

In [ ]:
import copy

In [ ]:


by_metric = {}
for metric in ["maximum_logprob", "average_logprob"]:
    by_metric[metric] = {}
    # ylabels = {
    #     "maximum_logprob": "Maximum log probability", 
    #     "perplexity": "Perplexity",
    #     "average_logprob": "Average log probability"
    # }
    # ylabel = ylabels[metric]
    sorted_seed_responses = {}
    for model in by_seed.keys():
        for seed,seed_dict in by_seed[model].items():
            sorted_seed_responses[seed] = {}


            # sort correct/wrong properly, by class
            correct = {}
            wrong = {}
            for cw_dict, cw_str in [[correct, "correct"], [wrong, "wrong"]]:
                # build cw dict
                ld = seed_dict[cw_str]
                for label in ld.keys():
                    if label not in cw_dict:
                        cw_dict[label] = []
                    for el in ld[label]:
                        cw_dict[label] += [el["uncertainty"][metric]]

                # save cw dict
                sorted_seed_responses[seed][cw_str] = cw_dict
        print(metric)
        by_metric[metric][model] = copy.deepcopy(sorted_seed_responses)
        # fig, ax = plot_seed_boxplots(sorted_seed_responses)
        # plt.rcParams.update({'font.size': 23})
        # ax.set_ylim(bottom=0)
        # ax.set_ylabel(ylabel, weight="bold")
        # ax.legend(loc="lower right")
        # ax.set_xlabel(f"{model} prediction", weight="bold")
        # fig.tight_layout()
        # # fig.show()
        
        # os.makedirs("out/figs/resubmission/by_seed/", exist_ok=True)
        # save_name = f"out/figs/resubmission/by_seed/{model}_{metric}_by_seed.pdf"
        # fig.savefig(save_name)

maximum_logprob
maximum_logprob
maximum_logprob
maximum_logprob
maximum_logprob
maximum_logprob
maximum_logprob
maximum_logprob
maximum_logprob
maximum_logprob
maximum_logprob
average_logprob
average_logprob
average_logprob
average_logprob
average_logprob
average_logprob
average_logprob
average_logprob
average_logprob
average_logprob
average_logprob


In [ ]:
from scipy.stats import mannwhitneyu
from tabulate import tabulate


table = []

pval_dict = dict([(model, {}) for model in list(by_metric.values())[0].keys()])
for model, model_dict in pval_dict.items():
    pval_dict[model] = {
        "Entailment": {},
        "Contradiction": {},
        "NotMentioned": {}
    }
    for label, label_dict in pval_dict[model].items():
        pval_dict[model][label] = dict((metric, {}) for metric in sorted(list(by_metric.keys())))
# print(pval_dict)


def mannwhitney_with_effect_size(x, y, alternative="less"):
    result = mannwhitneyu(x, y, alternative=alternative)
    n1, n2 = len(x), len(y)
    r = 1 - (2 * result.statistic) / (n1 * n2)
    return r, result.pvalue

for metric, metric_dict in by_metric.items():
    for model, model_dict in metric_dict.items():
        for seed, seed_dict in model_dict.items():
            labels = list(seed_dict["correct"].keys())
            for label in labels:
                correct_dist = seed_dict["correct"][label]
                wrong_dist = seed_dict["wrong"][label]
                effect_size, p = mannwhitney_with_effect_size(correct_dist, wrong_dist, alternative="less")
                # print(f"{metric}-{model}-{label}-{seed}:", stat, p)
                table.append([metric, model, label, seed, p, effect_size])
                pval_dict[model][label][metric][seed] = [round(p, 3), effect_size]

In [ ]:
import pandas as pd

records = [
    {"model": model, "label": label, "metric": metric, "seed": seed, "pval": pval, "effect_size": effect_size}
    for model, labels in pval_dict.items()
    for label, metrics in labels.items()
    for metric, seeds in metrics.items()
    for seed, (pval, effect_size) in seeds.items()
]

df = pd.DataFrame(records)


assert len(df) == 198, len(df)
df


,model,label,metric,seed,pval,effect_size
0,gemma4_e2b,Entailment,average_logprob,0,1.000,-0.207364
1,gemma4_e2b,Entailment,average_logprob,1,1.000,-0.189645
2,gemma4_e2b,Entailment,average_logprob,42,1.000,-0.154376
3,gemma4_e2b,Entailment,maximum_logprob,0,1.000,-0.406348
4,gemma4_e2b,Entailment,maximum_logprob,1,1.000,-0.400033
...,...,...,...,...,...,...
193,ministral-3_3b,NotMentioned,average_logprob,1,0.644,-0.077441
194,ministral-3_3b,NotMentioned,average_logprob,42,0.491,0.006237
195,ministral-3_3b,NotMentioned,maximum_logprob,0,0.262,0.125000
196,ministral-3_3b,NotMentioned,maximum_logprob,1,0.512,-0.003367


In [ ]:

from scipy.stats import false_discovery_control

df['q_adj'] = false_discovery_control(df['pval'].values, method='bh')
df['significant'] = df['q_adj'] <= 0.05

# with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    # print(df)


In [ ]:
def df_to_nested_dict_with_flag(df):
    nested = {}
    for row in df.itertuples(index=False):
        nested.setdefault(row.model, {}) \
              .setdefault(row.label, {}) \
              .setdefault(row.metric, {})[row.seed] = {
                  "q_adj": row.q_adj,
                  "significant": row.significant,
                  "effect_size": row.effect_size
              }
    return nested

nested = df_to_nested_dict_with_flag(df)
nested

{'gemma4_e2b': {'Entailment': {'average_logprob': {0: {'q_adj': 1.0,
     'significant': False,
     'effect_size': -0.20736385552901604},
    1: {'q_adj': 1.0,
     'significant': False,
     'effect_size': -0.1896446078431373},
    42: {'q_adj': 1.0,
     'significant': False,
     'effect_size': -0.15437604478700373}},
   'maximum_logprob': {0: {'q_adj': 1.0,
     'significant': False,
     'effect_size': -0.40634830491016505},
    1: {'q_adj': 1.0,
     'significant': False,
     'effect_size': -0.40003342245989315},
    42: {'q_adj': 1.0,
     'significant': False,
     'effect_size': -0.39628420450338253}}},
  'Contradiction': {'average_logprob': {0: {'q_adj': 0.47793103448275864,
     'significant': False,
     'effect_size': 0.08333333333333337},
    1: {'q_adj': 0.43489285714285714,
     'significant': False,
     'effect_size': 0.09296597025818809},
    42: {'q_adj': 0.6842647058823529,
     'significant': False,
     'effect_size': 0.05614891669209643}},
   'maximum_logprob'

In [ ]:
# \multirow{3}{*}{Model A}
#     & Entailment & val & val & val & val & val & val & val & val & val  \\
#     & Contradiction& val & val & val & val & val & val & val & val & val  \\
#     & NotMentioned & val & val & val & val & val & val & val & val & val  \\
tables_sorted = {}
with open(f"mannwhitney-less_{set_logprob_scope}.txt", "w") as f:
    for model, model_dict in nested.items():
        model_text = " ".join(model.split("_")).capitalize()[:-1] + "B"
        if "Gemma" in model_text and model_text[-3] == "e":
            model_text = model_text[:-3] + "E" + model_text[-2:]
        tables_sorted[model_text] = {}
        
        for metric in ["average_logprob", "maximum_logprob"]:
            table_buffer = ""
            metric_text = " ".join(metric.split("_")).capitalize()
            # caption = f"{metric_text} for {model_text}"
            caption = metric_text
            table_buffer += ("""
\\begin{tabular}{l|c||c|c}
Label & Seed & $\\tilde{p}_i$ & Effect size \\\\
\\hline
\\hline

""")
            for label in ["Entailment", "Contradiction", "NotMentioned"]:
                metric_dict = model_dict[label][metric]
                table_buffer += ("\\multirow{3}{*}{" + label + "}\n")
                for seed in [0, 1, 42]:
                    table_buffer += (f"\t & {seed}")
                    vals = ""
                    # print(metric, end=" ")
                    vals += " & "
                    p = round(metric_dict[seed]['q_adj'], 3)
                    significant = metric_dict[seed]['significant']
                    e_size = round(metric_dict[seed]['effect_size'], 3)
                    # print(p, end=" ")
                    if significant:
                        vals += "\\cellcolor{SpringGreen} "

                    if p < 0.01:
                        vals += " $\\tilde{p}_i < 0.01$"
                    else:
                        vals += f" {p}"
    
                    vals += f" & {e_size}"
                    table_buffer += (vals)
                    table_buffer += (" \\\\\n")
                    # print("")
                # table_buffer += ("\hline\n")
                table_buffer += "\hline \n "
            table_buffer += ("\\end{tabular} \\caption{" + caption + "}") 

            tables_sorted[model_text][metric_text] = table_buffer
            # print(table_buffer)
# print(tabulate(nested, headers=["Metric", "Model", "Label", "Seed", "Stat", "P"]))
print(tables_sorted)

{'Gemma4 E2B': {'Average logprob': '\n\\begin{tabular}{l|c||c|c}\nLabel & Seed & $\\tilde{p}_i$ & Effect size \\\\\n\\hline\n\\hline\n\n\\multirow{3}{*}{Entailment}\n\t & 0 &  1.0 & -0.207 \\\\\n\t & 1 &  1.0 & -0.19 \\\\\n\t & 42 &  1.0 & -0.154 \\\\\n\\hline \n \\multirow{3}{*}{Contradiction}\n\t & 0 &  0.478 & 0.083 \\\\\n\t & 1 &  0.435 & 0.093 \\\\\n\t & 42 &  0.684 & 0.056 \\\\\n\\hline \n \\multirow{3}{*}{NotMentioned}\n\t & 0 &  1.0 & -0.035 \\\\\n\t & 1 &  1.0 & -0.044 \\\\\n\t & 42 &  1.0 & -0.116 \\\\\n\\hline \n \\end{tabular} \\caption{Average logprob}', 'Maximum logprob': '\n\\begin{tabular}{l|c||c|c}\nLabel & Seed & $\\tilde{p}_i$ & Effect size \\\\\n\\hline\n\\hline\n\n\\multirow{3}{*}{Entailment}\n\t & 0 &  1.0 & -0.406 \\\\\n\t & 1 &  1.0 & -0.4 \\\\\n\t & 42 &  1.0 & -0.396 \\\\\n\\hline \n \\multirow{3}{*}{Contradiction}\n\t & 0 &  1.0 & -0.318 \\\\\n\t & 1 &  1.0 & -0.235 \\\\\n\t & 42 &  1.0 & -0.208 \\\\\n\\hline \n \\multirow{3}{*}{NotMentioned}\n\t & 0 & \\cell

In [ ]:
with open(f"mannwhitney-less_{set_logprob_scope}.txt", "w") as f:
    for model in sorted(list(tables_sorted.keys())):
        f.write("\\begin{table*}[htbp] \\centering")
        for metrics in sorted(list(list(tables_sorted.values())[0].keys())):
            f.write("\\begin{subtable}{0.48\\textwidth}")
            print(model, metrics)
            f.write(tables_sorted[model][metrics])
            f.write("\\end{subtable}")
        f.write("\\caption{" + model + "}")
        f.write("\\end{table*}\n")

Gemma4 26B Average logprob
Gemma4 26B Maximum logprob
Gemma4 E2B Average logprob
Gemma4 E2B Maximum logprob
Gemma4 E4B Average logprob
Gemma4 E4B Maximum logprob
Ministral-3 14B Average logprob
Ministral-3 14B Maximum logprob
Ministral-3 3B Average logprob
Ministral-3 3B Maximum logprob
Ministral-3 8B Average logprob
Ministral-3 8B Maximum logprob
Qwen3.5 27B Average logprob
Qwen3.5 27B Maximum logprob
Qwen3.5 4B Average logprob
Qwen3.5 4B Maximum logprob
Qwen3.5 4b thinB Average logprob
Qwen3.5 4b thinB Maximum logprob
Qwen3.5 9B Average logprob
Qwen3.5 9B Maximum logprob
Qwen3.6 35B Average logprob
Qwen3.6 35B Maximum logprob


In [ ]:
# # \multirow{3}{*}{Model A}
# #     & Entailment & val & val & val & val & val & val & val & val & val  \\
# #     & Contradiction& val & val & val & val & val & val & val & val & val  \\
# #     & NotMentioned & val & val & val & val & val & val & val & val & val  \\
# with open(f"mannwhitney-less_{set_logprob_scope}.txt", "w") as f:
#     for model, model_dict in nested.items():
#         f.write("\multirow{3}{*}{" + model + "}\n")
#         for label, label_dict in model_dict.items():
#             f.write(f"\t & {label}")
#             for metric, metric_dict in label_dict.items():
#                 vals = ""
#                 print(metric, end=" ")
#                 for seed in [0, 1, 42]:
#                     vals += " & "
#                     p = round(metric_dict[seed]['q_adj'],3)
#                     significant = metric_dict[seed]['significant']
#                     e_size = round(metric_dict[seed]['effect_size'], 3)
#                     print(p, end=" ")
#                     if significant:
#                         vals += "\cellcolor{SpringGreen} "

#                     if p < 0.01:
#                         vals += " $\\tilde{p}_i < 0.01$"
#                     else:
#                         vals += f" {p}"

#                     vals += f" & {e_size}"
                    
#                 f.write(vals)
#             print("")
#             f.write(" \\\\\n")
#         f.write("\hline\n")
# # print(tabulate(nested, headers=["Metric", "Model", "Label", "Seed", "Stat", "P"]))

In [ ]:
# from scipy.stats import mannwhitneyu

# def mannwhitney_with_effect_size(x, y):
#     result = mannwhitneyu(x, y, alternative='less')
#     n1, n2 = len(x), len(y)
#     r = 1 - (2 * result.statistic) / (n1 * n2)
#     return result.pvalue, r

# records = []
# for metric, metric_dict in by_metric.items():
#     for model, model_dict in metric_dict.items():
#         for seed, seed_dict in model_dict.items():
#             labels = list(seed_dict["correct"].keys())
#             for label in labels:
#                 correct_dist = seed_dict["correct"][label]
#                 wrong_dist = seed_dict["wrong"][label]
#                 pval, r = mannwhitney_with_effect_size(correct_dist, wrong_dist)
#                 records.append({
#                     "model": model, "label": label, "metric": metric,
#                     "seed": seed, "pval": pval, "effect_size": r
#                 })